# Uncertainty Shocks

The model of [Di Tella (2017)](https://doi.org/10.1086/694290), adapted from an example of EconPDEs.jl. Experts and households have recursive preferences, and an agency friction (moral hazard) forces experts to bear idiosyncratic risk. There are two states: the experts' wealth share `x`, and a stochastic idiosyncratic risk `nu`, the uncertainty shock.

Three functions are solved together: `pA` and `pB` for experts and households, and the capital price `p`, set by an algebraic market-clearing condition and not by its own time derivative.

In [ ]:
Pollis.packages("EconPDEs", "Plots")

## From EconPDEs.jl

The code is the package's example, with Greek letters written in Latin (`gamma`, `psi`, `rho`, `tau`, `delta`, `sigma`, `phi`, `nu`, `kappa`).

### The model

The investment technology is `i(g) = A (g + delta)^2 + B (g + delta)` with `delta = 0.05`. According to the package's example, the default `A` and the implied `B` make the benchmark point near `x = 0.10` and `nu = 0.25` reproduce the paper's targets, growth of about 0.02 and investment of about 0.20.

In [ ]:
using EconPDEs, Plots, Printf

Base.@kwdef struct DiTellaModel
  # Utility Function
  gamma::Float64 = 5.0        # relative risk aversion
  psi::Float64 = 2.0        # elasticity of intertemporal substitution
  rho::Float64 = 0.0665     # rate of time preference (discount rate)
  tau::Float64 = 1.15       # transition rate between experts and households

  # Technology
  A::Float64 = 54.0       # investment adjustment-cost parameter calibrated to the paper targets
  delta::Float64 = 0.05       # depreciation rate in the investment technology
  B::Float64 = (0.20 - A * (0.02 + delta)^2) / (0.02 + delta)  # matches i(0.02) = 0.20
  sigma::Float64 = 0.0125     # aggregate (fundamental) volatility of capital

  # MoralHazard
  phi::Float64 = 0.2        # moral-hazard idiosyncratic-risk retention parameter

  # Idiosyncratic
  nubar::Float64 = 0.25    # long-run mean of idiosyncratic risk state nu
  kappanu::Float64 = 1.38      # mean-reversion speed of nu (uncertainty process)
  sigmanubar::Float64 = -0.17  # volatility loading of the nu process
end

In [ ]:
m = DiTellaModel()

### The grid and the guess

The grid has a key for each state and includes the slices used in the plot: `x` at 0.05, 0.10 and 0.20, and `nu` at 0.10, 0.25 and 0.60. In the guess, `1 / pA` and `1 / pB` are the two agents' consumption rates, and the relative investment opportunity is `Q = (pA / pB)^(1 / (psi - 1))`.

In [ ]:
function initialize_stategrid(m::DiTellaModel; xn = 30, nun = 30)
  xs = sort(unique(vcat(collect(range(0.01, 0.99, length = xn)), [0.05, 0.10, 0.20])))
  nus = sort(unique(vcat(collect(range(0.01, 1.00, length = nun)), [0.10, 0.25, 0.60])))
  (; x = xs, nu = nus)
end

In [ ]:
function initialize_guess(m::DiTellaModel, stategrid)
  xn = length(stategrid[:x])
  nun = length(stategrid[:nu])
  (; pA = ones(xn, nun), pB = ones(xn, nun), p = ones(xn, nun))
end

In [ ]:
stategrid = initialize_stategrid(m)
guess = initialize_guess(m, stategrid)

### The equations

With two states, the first derivatives are upwinded on the signs of their drifts, and the cross derivative on the sign of the covariance of `x` and `nu`.

In [ ]:
function (m::DiTellaModel)(state::NamedTuple, u::NamedTuple)
  (; gamma, psi, rho, tau, A, delta, B, sigma, phi, nubar, kappanu, sigmanubar) = m
  (; x, nu) = state
  (; pA, pAx_up, pAx_down, pAnu_up, pAnu_down, pAxx, pAxnu_up, pAxnu_down, pAnunu, pB, pBx_up, pBx_down, pBnu_up, pBnu_down, pBxx, pBxnu_up, pBxnu_down, pBnunu, p, px_up, px_down, pnu_up, pnu_down, pxx, pxnu_up, pxnu_down, pnunu) = u

  # drift and volatility of state variable nu
  q = (p - B) / (2 * A)
  g = q - delta
  i = A * q^2 + B * q
  munu = kappanu * (nubar - nu)
  sigmanu = sigmanubar * sqrt(nu)
  pAnu = (munu >= 0) ? pAnu_up : pAnu_down
  pBnu = (munu >= 0) ? pBnu_up : pBnu_down
  pnu = (munu >= 0) ? pnu_up : pnu_down

  pAx, pBx, px = pAx_up, pBx_up, px_up
  iter = 0
  @label start
  sigmaX = x * (1 - x) * (1 - gamma) / (gamma * (psi - 1)) * (pAnu / pA - pBnu / pB) * sigmanu / (1 - x * (1 - x) * (1 - gamma) / (gamma * (psi - 1)) * (pAx / pA - pBx / pB))
  sigmapA = pAx / pA * sigmaX + pAnu / pA * sigmanu
  sigmapB = pBx / pB * sigmaX + pBnu / pB * sigmanu
  sigmap = px / p * sigmaX + pnu / p * sigmanu
  kappa = (sigmap + sigma - (1 - gamma) / (gamma * (psi - 1)) * (x * sigmapA + (1 - x) * sigmapB)) / (1 / gamma)
  # kappaidio is the price of idiosyncratic risk (distinct from the parameter kappanu, the mean-reversion speed of nu).
  kappaidio = gamma * phi * nu / x
  sigmaA = kappa / gamma + (1 - gamma) / (gamma * (psi - 1)) * sigmapA
  nuA = kappaidio / gamma
  sigmaB = kappa / gamma + (1 - gamma) / (gamma * (psi - 1)) * sigmapB

  # Interest rate r
  muX = x * (1 - x) * ((sigmaA * kappa + nuA * kappaidio - 1 / pA - tau) - (sigmaB * kappa -  1 / pB + tau * x / (1 - x)) - (sigmaA - sigmaB) * (sigma + sigmap))

  # upwinding
  if (iter == 0) && (muX <= 0)
    iter += 1
    pAx, pBx, px = pAx_down, pBx_down, px_down
    @goto start
  end

  # upwind the cross derivative on the sign of its coefficient sigmaX * sigmanu (the x-nu covariance)
  pAxnu = (sigmaX * sigmanu >= 0) ? pAxnu_up : pAxnu_down
  pBxnu = (sigmaX * sigmanu >= 0) ? pBxnu_up : pBxnu_down
  pxnu = (sigmaX * sigmanu >= 0) ? pxnu_up : pxnu_down

  mupA = pAx / pA * muX + pAnu / pA * munu + 0.5 * pAxx / pA * sigmaX^2 + 0.5 * pAnunu / pA * sigmanu^2 + pAxnu / pA * sigmaX * sigmanu
  mupB = pBx / pB * muX + pBnu / pB * munu + 0.5 * pBxx / pB * sigmaX^2 + 0.5 * pBnunu / pB * sigmanu^2 + pBxnu / pB * sigmaX * sigmanu
  mup = px / p * muX + pnu / p * munu + 0.5 * pxx / p * sigmaX^2 + 0.5 * pnunu / p * sigmanu^2 + pxnu / p * sigmaX * sigmanu
  r = (1 - i) / p + g + mup + sigma * sigmap - kappa * (sigma + sigmap) - gamma / x * (phi * nu)^2

  # Market Pricing
  pAt = - pA * (1 / pA  + (psi - 1) * tau / (1 - gamma) * ((pA / pB)^((1 - gamma) / (1 - psi)) - 1) - psi * rho + (psi - 1) * (r + kappa * sigmaA + kappaidio * nuA) + mupA - (psi - 1) * gamma / 2 * (sigmaA^2 + nuA^2) + (2 - psi - gamma) / (2 * (psi - 1)) * sigmapA^2 + (1 - gamma) * sigmapA * sigmaA)
  pBt = - pB * (1 / pB - psi * rho + (psi - 1) * (r + kappa * sigmaB) + mupB - (psi - 1) * gamma / 2 * sigmaB^2 + (2 - psi - gamma) / (2 * (psi - 1)) * sigmapB^2 + (1 - gamma) * sigmapB * sigmaB)
  # algebraic constraint
  pt = - p * ((1 - i) / p - x / pA - (1 - x) / pB)
  Q = (pA / pB)^(1 / (psi - 1))
  return (; pAt, pBt, pt), (; sigmax = sigmaX, Q, r, kappa, sigmap, growth = g, investment = i)
end

### Solving it

`p` enters as an algebraic variable, which `is_algebraic` tells `pdesolve`.

In [ ]:
result = pdesolve(m, stategrid, guess; is_algebraic = (; pA = false, pB = false, p = true))

### The solution

The package's example follows the panels of Figure 1 of Di Tella's paper: the capital price `p`, the volatility of the experts' wealth share `sigmax` and the relative investment opportunity `Q`, each against `nu` at fixed `x` and against `x` at fixed `nu`.

An uncertainty shock raises the idiosyncratic risk experts must retain. When their balance sheets are weak, at low `x`, the same rise in `nu` lowers the capital price more.

In [ ]:
xs = stategrid[:x]
nus = stategrid[:nu]
p = result.solution.p
sigmax = result.saved.sigmax
Q = result.saved.Q
line_styles = [:solid, :dot, :dash]
x_slices = [0.05, 0.10, 0.20]
nu_slices = [0.10, 0.25, 0.60]

pnu = plot(; xlabel = "risk state nu", ylabel = "capital price p",
          xlims = (0, 1), ylims = (5.6, 7.2), legend = :bottomleft)
sigmaxnu = plot(; xlabel = "risk state nu", ylabel = "volatility sigmax",
           xlims = (0, 1), ylims = (0, 0.01), legend = false)
Qnu = plot(; xlabel = "risk state nu", ylabel = "relative opportunities Q",
          xlims = (0, 1), ylims = (1.0, 2.7), legend = false)
for (x_slice, line_style) in zip(x_slices, line_styles)
  x_index = findfirst(==(x_slice), xs)
  plot!(pnu, nus, p[x_index, :]; label = @sprintf("x = %.2f", x_slice),
        color = :black, linestyle = line_style)
  plot!(sigmaxnu, nus, sigmax[x_index, :]; color = :black, linestyle = line_style)
  plot!(Qnu, nus, Q[x_index, :]; color = :black, linestyle = line_style)
end
px = plot(; xlabel = "wealth share x", ylabel = "capital price p",
          xlims = (0, 1), ylims = (5.6, 7.2), legend = :bottomright)
sigmaxx = plot(; xlabel = "wealth share x", ylabel = "volatility sigmax",
           xlims = (0, 1), ylims = (0, 0.01), legend = false)
Qx = plot(; xlabel = "wealth share x", ylabel = "relative opportunities Q",
          xlims = (0, 1), ylims = (1.0, 2.7), legend = false)
for (nu_slice, line_style) in zip(nu_slices, line_styles)
  nu_index = findfirst(==(nu_slice), nus)
  plot!(px, xs, p[:, nu_index]; label = @sprintf("nu = %.2f", nu_slice),
        color = :black, linestyle = line_style)
  plot!(sigmaxx, xs, sigmax[:, nu_index]; color = :black, linestyle = line_style)
  plot!(Qx, xs, Q[:, nu_index]; color = :black, linestyle = line_style)
end
plot(pnu, sigmaxnu, Qnu, px, sigmaxx, Qx; layout = (2, 3), size = (940, 560),
     left_margin = 8Plots.mm, bottom_margin = 8Plots.mm, right_margin = 4Plots.mm)

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/di_tella.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

Di Tella, S. (2017). Uncertainty shocks and balance sheet recessions. *Journal of Political Economy* 125(6), 2038–2081. [doi:10.1086/694290](https://doi.org/10.1086/694290)